# v17: kriging prior (構造深度 geo_z の空間補間)

フォーラムの『LB<7 の magic』= **baseline を直線dipではなく kriging にする**。

- `geo_z = TVT + Z`(構造絶対深度)。`TVT = geo_z - Z`。
- **kriging prior**: 対象坑井の評価点 (X,Y) における `geo_z` を、**他の坑井の (X,Y, geo_z) から空間補間**
  (局所RBF重み付き1次=平面フィット → 構造dipを自然に表現)。対象坑井自身は除外(リーク防止)。
- **オフセット除去('zero at PS')**: 既知ゾーン末尾で krige を真値に合わせて絶対レベルを校正。
- `baseline_tvt = geo_z_prior - Z`。これを直線dip baseline と比較。

確認: (a) **baseline単体 overall RMSE**(候補なし)が直線(≈15)からどれだけ下がるか, (b) 可視化, (c) krige残差まわりの oracle。

> test互換: test も X,Y を持つ。隣人=train坑井, query=対象坑井(X,Y), オフセットは対象の既知ゾーンから。
> CV厳密化: prior は本来 fold-train の坑井だけで作るべき(ここは leave-one-well-out で全train使用=やや楽観)。

In [1]:
import os, pickle, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import nn_pf_common as C
importlib.reload(C)

# X,Y を含めて読み直す(nn_pf_common は X,Y を読んでいないため)
def build_well_xy(wid, split='train', fit_n=300):
    hw, tw = C.load_well_raw(wid, split)
    md = hw['MD'].to_numpy(float); x = hw['X'].to_numpy(float); y = hw['Y'].to_numpy(float)
    z = hw['Z'].to_numpy(float)
    gr = pd.Series(hw['GR'].to_numpy(float)).interpolate(limit_direction='both').to_numpy()
    tin = hw['TVT_input'].to_numpy(float)
    has_true = 'TVT' in hw.columns
    tvt = hw['TVT'].to_numpy(float) if has_true else np.full_like(md, np.nan)
    known = np.isfinite(tin); evalm = ~known
    geo_z_known = tin + z                      # 既知ゾーンの geo_z
    geo_z_true = (tvt + z) if has_true else np.full_like(md, np.nan)
    # 直線dip baseline(比較用)
    kmd, ks = md[known], geo_z_known[known]
    if kmd.size >= 2:
        sel = slice(max(0, kmd.size - fit_n), kmd.size)
        A = np.vstack([kmd[sel], np.ones(kmd[sel].size)]).T
        a, b = np.linalg.lstsq(A, ks[sel], rcond=None)[0]
    else:
        a, b = 0.0, (ks.mean() if ks.size else 0.0)
    geo_z_lin = a * md + b
    tw = tw.sort_values('TVT')
    return dict(wid=wid, md=md, x=x, y=y, z=z, gr=gr, tin=tin, tvt=tvt,
                known=known, evalm=evalm, geo_z_known=geo_z_known, geo_z_true=geo_z_true,
                geo_z_lin=geo_z_lin, has_true=has_true,
                tw_tvt=tw['TVT'].to_numpy(float),
                tw_gr=pd.Series(tw['GR'].to_numpy(float)).interpolate(limit_direction='both').to_numpy())

WELLS = [build_well_xy(w, 'train') for w in C.list_well_ids('train')]
print('wells:', len(WELLS))

wells: 773


In [ ]:
# ===== 空間クラウド(全坑井の X,Y,geo_z を間引いて集約) =====
# train は全行 geo_z 既知(=TVT+Z)。対象坑井は query 時に除外する。
STRIDE = 20
Xc, Yc, Gc, Wc = [], [], [], []
for wi, w in enumerate(WELLS):
    g = w['geo_z_true']
    m = np.isfinite(g) & np.isfinite(w['x']) & np.isfinite(w['y'])
    idx = np.where(m)[0][::STRIDE]
    Xc.append(w['x'][idx]); Yc.append(w['y'][idx]); Gc.append(g[idx])
    Wc.append(np.full(idx.size, wi))
Xc = np.concatenate(Xc); Yc = np.concatenate(Yc); Gc = np.concatenate(Gc); Wc = np.concatenate(Wc)
COORDS = np.stack([Xc, Yc], 1)
print('cloud points:', COORDS.shape[0])

def krige_predict(coords_tr, g_tr, q, k=48, ls_factor=1.0, reg=1e-2, degree=1, clamp=True):
    """局所kriging。degree=0:重み付き平均(安全), degree=1:平面フィット(dip勾配を表現)。
    数値安定化: dx,dy を局所長さ ls で正規化してから解く(予測は query位置の切片=スケール不変)。"""
    tree = cKDTree(coords_tr)
    k = min(k, coords_tr.shape[0])
    dist, idx = tree.query(q, k=k)
    if k == 1:
        return g_tr[idx]
    nb = coords_tr[idx]                       # [Q,k,2]
    gv = g_tr[idx]                            # [Q,k]
    ls = ls_factor * dist.mean(1, keepdims=True) + 1e-6
    wgt = np.exp(-(dist ** 2) / (2 * ls ** 2)) + 1e-12
    if degree == 0:
        pred = (wgt * gv).sum(1) / wgt.sum(1)
    else:
        dx = (nb[..., 0] - q[:, None, 0]) / ls   # ★スケール化(条件数改善)
        dy = (nb[..., 1] - q[:, None, 1]) / ls
        Adesign = np.stack([np.ones_like(dx), dx, dy], -1)   # [Q,k,3]
        WA = Adesign * wgt[..., None]
        AtA = np.einsum('qki,qkj->qij', WA, Adesign)
        tr = np.einsum('qii->q', AtA)[:, None, None] / 3.0
        AtA = AtA + reg * tr * np.eye(3)[None]               # トレース比例ridge
        Atg = np.einsum('qki,qk->qi', WA, gv)
        coef = np.linalg.solve(AtA, Atg[..., None])[..., 0]  # numpy2.0: ベクトル次元を明示
        pred = coef[:, 0]                                    # dx=dy=0 の切片 = 予測
    if clamp:
        lo = gv.min(1); hi = gv.max(1); rng_ = (hi - lo)
        pred = np.clip(pred, lo - 0.5 * rng_, hi + 0.5 * rng_)  # 近傍値域±50%に制限(暴走防止)
    return pred

In [ ]:
# ===== 各坑井の kriging prior を計算(対象坑井を除外, 既知ゾーンでオフセット校正) =====
# OFFSET_TAIL は上の sanity セルで定義済み。
def compute_krige_prior(wi, k=48, ls_factor=1.0, degree=1, reg=1e-2):
    w = WELLS[wi]
    sub = (Wc != wi)                                   # 対象坑井を除外(リーク防止)
    coords_tr = COORDS[sub]; g_tr = Gc[sub]
    ev = np.where(w['evalm'])[0]
    kn = np.where(w['known'])[0]
    kn_tail = kn[-OFFSET_TAIL:] if kn.size else kn
    qi = np.concatenate([ev, kn_tail])
    q = np.stack([w['x'][qi], w['y'][qi]], 1)
    pred = krige_predict(coords_tr, g_tr, q, k=k, ls_factor=ls_factor, degree=degree, reg=reg)
    pred_ev = pred[:ev.size]; pred_tail = pred[ev.size:]
    # オフセット: 既知末尾で krige を真の geo_z に合わせる('zero at PS')
    offset = np.mean(w['geo_z_known'][kn_tail] - pred_tail) if kn_tail.size else 0.0
    return ev, pred_ev + offset, offset

# 全坑井で計算してキャッシュ。★直後は壊れたキャッシュ回避のため use_cache=False で再生成すること。
def run_all_priors(n_wells=None, k=48, ls_factor=1.0, degree=1, reg=1e-2,
                   cache='artifacts_v17/krige_prior.pkl', use_cache=True):
    if use_cache and os.path.exists(cache):
        with open(cache, 'rb') as f:
            return pickle.load(f)
    res = {}
    idxs = range(len(WELLS) if n_wells is None else n_wells)
    for j, wi in enumerate(idxs):
        ev, gp, off = compute_krige_prior(wi, k=k, ls_factor=ls_factor, degree=degree, reg=reg)
        res[WELLS[wi]['wid']] = dict(ev=ev, geo_z_prior=gp, offset=off)
        if (j + 1) % 50 == 0:
            print(f'  {j+1} wells')
    if cache and (n_wells is None):
        os.makedirs(os.path.dirname(cache), exist_ok=True)
        with open(cache, 'wb') as f:
            pickle.dump(res, f)
    return res

# sanity で良かった (degree, ls_factor) をここに反映して回す。初回は use_cache=False。
PRIORS = run_all_priors(n_wells=None, k=48, ls_factor=1.0, degree=1, use_cache=False)
print('priors:', len(PRIORS))

In [3]:
# ===== 各坑井の kriging prior を計算(対象坑井を除外, 既知ゾーンでオフセット校正) =====
OFFSET_TAIL = 200   # オフセット校正に使う既知ゾーン末尾の点数

def compute_krige_prior(wi, k=48, ls_factor=1.0):
    w = WELLS[wi]
    sub = (Wc != wi)                                   # 対象坑井を除外(リーク防止)
    coords_tr = COORDS[sub]; g_tr = Gc[sub]
    # 評価点 + 既知末尾(オフセット用)をまとめて予測
    ev = np.where(w['evalm'])[0]
    kn = np.where(w['known'])[0]
    kn_tail = kn[-OFFSET_TAIL:] if kn.size else kn
    qi = np.concatenate([ev, kn_tail])
    q = np.stack([w['x'][qi], w['y'][qi]], 1)
    pred = krige_predict(coords_tr, g_tr, q, k=k, ls_factor=ls_factor)
    pred_ev = pred[:ev.size]; pred_tail = pred[ev.size:]
    # オフセット: 既知末尾で krige を真の geo_z に合わせる('zero at PS')
    if kn_tail.size:
        offset = np.mean(w['geo_z_known'][kn_tail] - pred_tail)
    else:
        offset = 0.0
    geo_z_prior_ev = pred_ev + offset
    return ev, geo_z_prior_ev, offset

# 全坑井で計算してキャッシュ(初回のみ重い)。n_wells で小さく試せる。
def run_all_priors(n_wells=None, k=48, ls_factor=1.0, cache='artifacts_v17/krige_prior.pkl', use_cache=True):
    if use_cache and os.path.exists(cache):
        with open(cache, 'rb') as f:
            return pickle.load(f)
    res = {}
    idxs = range(len(WELLS) if n_wells is None else n_wells)
    for j, wi in enumerate(idxs):
        ev, gp, off = compute_krige_prior(wi, k=k, ls_factor=ls_factor)
        res[WELLS[wi]['wid']] = dict(ev=ev, geo_z_prior=gp, offset=off)
        if (j + 1) % 50 == 0:
            print(f'  {j+1} wells')
    if cache and (n_wells is None):
        os.makedirs(os.path.dirname(cache), exist_ok=True)
        with open(cache, 'wb') as f:
            pickle.dump(res, f)
    return res

PRIORS = run_all_priors(n_wells=None, k=48, ls_factor=1.0, use_cache=True)
print('priors:', len(PRIORS))

ValueError: solve: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (m,m),(m,n)->(m,n) (size 4036 is different from 3)

In [ ]:
# ===== baseline単体 overall RMSE: 直線dip vs kriging (候補なし) =====
# これが magic の効き。直線(≈15)から kriging でどれだけ下がるか。
def baseline_overall(use='krige'):
    errs = []
    for w in WELLS:
        if not w['has_true']:
            continue
        ev = np.where(w['evalm'])[0]
        t = w['tvt'][ev]; m = np.isfinite(t)
        if not m.any():
            continue
        if use == 'krige':
            pr = PRIORS[w['wid']]
            base_tvt = pr['geo_z_prior'] - w['z'][ev]
        else:  # linear
            base_tvt = w['geo_z_lin'][ev] - w['z'][ev]
        errs.append((base_tvt - t)[m])
    e = np.concatenate(errs)
    return float(np.sqrt(np.mean(e ** 2)))

print(f'baseline-only overall RMSE  | linear-dip = {baseline_overall("linear"):.3f}')
print(f'baseline-only overall RMSE  | KRIGING    = {baseline_overall("krige"):.3f}')
print('  -> kriging が大きく下がれば magic 成立。これに候補生成+選択を足すとさらに下がる余地。')

In [ ]:
# ===== 可視化: kriging prior vs 真値 vs 直線baseline =====
def plot_krige(key, ctx=400):
    w = WELLS[key] if isinstance(key, int) else next(x for x in WELLS if x['wid'] == key)
    pr = PRIORS[w['wid']]; ev = pr['ev']
    base_tvt_k = pr['geo_z_prior'] - w['z'][ev]
    base_tvt_l = w['geo_z_lin'][ev] - w['z'][ev]
    t = w['tvt'][ev]; m = np.isfinite(t)
    rmse_k = np.sqrt(np.mean((base_tvt_k[m] - t[m]) ** 2))
    rmse_l = np.sqrt(np.mean((base_tvt_l[m] - t[m]) ** 2))
    kn = np.where(w['known'])[0]; kt = kn[kn >= kn[-1] - ctx] if kn.size else kn
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
    # geo_z 空間
    ax[0].plot(ev, w['geo_z_true'][ev], 'k', lw=2, label='true geo_z')
    ax[0].plot(ev, pr['geo_z_prior'], 'r', lw=2, label='krige prior')
    ax[0].plot(ev, w['geo_z_lin'][ev], 'b', lw=1.5, label='linear dip')
    ax[0].set_title(f"{w['wid']} geo_z"); ax[0].legend(); ax[0].set_xlabel('row idx')
    # TVT 空間
    ax[1].plot(ev, t, 'k', lw=2, label='true TVT')
    ax[1].plot(ev, base_tvt_k, 'r', lw=2, label=f'krige rmse={rmse_k:.2f}')
    ax[1].plot(ev, base_tvt_l, 'b', lw=1.5, label=f'linear rmse={rmse_l:.2f}')
    if kt.size:
        ax[1].plot(kt, w['tin'][kt], 'g', lw=1.5, label='known TVT')
    ax[1].set_title('TVT'); ax[1].legend()
    # 空間配置(対象=赤, 周囲=薄灰)
    ax[2].scatter(Xc[::5], Yc[::5], s=2, c='lightgray')
    ax[2].plot(w['x'], w['y'], 'r', lw=2)
    ax[2].set_title('XY: target(red) vs cloud'); ax[2].set_aspect('equal', 'datalim')
    plt.tight_layout(); plt.show()
    print(f"{w['wid']}: krige_rmse={rmse_k:.2f}  linear_rmse={rmse_l:.2f}  offset={pr['offset']:.2f}")

for i in [0, 1, 2, 3]:
    plot_krige(i)

In [ ]:
# ===== (任意) krige baseline まわりの候補 oracle =====
# 残差 r = TVT_true - krige_baseline_tvt は小さいはず。0中心ランダムウォークでも oracle が下がるか確認。
def oracle_krige(P=512, K=6, step=6.0, seed=0):
    rng = np.random.default_rng(seed); errs = []
    frac = np.linspace(0, 1, K)
    for w in WELLS:
        if not w['has_true']:
            continue
        pr = PRIORS[w['wid']]; ev = pr['ev']
        base = pr['geo_z_prior'] - w['z'][ev]
        t = w['tvt'][ev]; m = np.isfinite(t)
        if not m.any():
            continue
        L = ev.size
        md = w['md'][ev]; u = (md - md[0]) / max(md[-1] - md[0], 1.0)
        inc = rng.standard_normal((P, K)) * step; vals = np.cumsum(inc, 1); vals[:, 0] = 0
        resid = np.stack([np.interp(u, frac, vals[p]) for p in range(P)])
        cand = base[None, :] + resid
        cand[0] = base
        d = cand[:, m] - t[None, m]
        errs.append(d[int(np.argmin(np.sqrt(np.mean(d ** 2, 1))))])
    e = np.concatenate(errs)
    print(f'oracle around KRIGE baseline (P={P}, step={step}) overall RMSE = {np.sqrt(np.mean(e**2)):.3f}')

oracle_krige(P=512, step=6.0)